# Entrenamiento Mechatronics AI en Colab

Entorno de ejecución → Cambiar tipo → **GPU T4**.

En Drive debe existir `MyDrive/mechatronics-ai/dataset/` con `data.yaml` y las carpetas `train/ val/ test/` generadas por `scripts/split_dataset.py`.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
BASE = '/content/drive/MyDrive/mechatronics-ai'

In [ ]:
!pip install -q ultralytics albumentations
import ultralytics; ultralytics.checks()

## Entrenar (transfer learning desde YOLO11n)

In [ ]:
import albumentations as A
from ultralytics import YOLO

# Augmentation de 'cámara barata' (igual que scripts/train.py)
aug = [
    A.OneOf([A.MotionBlur(blur_limit=(3, 9)), A.GaussianBlur(blur_limit=(3, 7)), A.Defocus(radius=(1, 3))], p=0.3),
    A.OneOf([A.GaussNoise(std_range=(0.02, 0.08)), A.ISONoise(color_shift=(0.01, 0.05), intensity=(0.1, 0.5))], p=0.3),
    A.ImageCompression(quality_range=(30, 80), p=0.3),
    A.Downscale(scale_range=(0.5, 0.9), p=0.2),
    A.RandomBrightnessContrast(brightness_limit=(-0.35, 0.15), contrast_limit=0.2, p=0.3),
    A.ToGray(p=0.05),
]

modelo = YOLO('yolo11n.pt')
modelo.train(data=f'{BASE}/dataset/data.yaml', epochs=80, imgsz=640, batch=32, patience=20,
             device=0, project=f'{BASE}/runs', name='mechatronics', exist_ok=True, augmentations=aug)

## Evaluar en test

In [ ]:
best = YOLO(f'{BASE}/runs/mechatronics/weights/best.pt')
res = best.val(data=f'{BASE}/dataset/data.yaml', split='test')
print('P', res.box.mp, 'R', res.box.mr, 'mAP50', res.box.map50, 'mAP50-95', res.box.map)

## Exportar

Descarga `best.pt` a `models/` en tu PC y exporta allí a OpenVINO con `python scripts/export.py --model models/best.pt` (la exportación OpenVINO conviene hacerla en la CPU destino).

In [ ]:
from google.colab import files
files.download(f'{BASE}/runs/mechatronics/weights/best.pt')